# 07 Dashboards and Genie

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Turn a Gold table into an <b>AI/BI dashboard</b> (datasets, visualizations, filters, parameters, sharing and scheduling) and a <b>Genie space</b> where business users ask questions in natural language. Learn what data engineers must provide to make both work well: clean, documented, well-modelled tables.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Dashboards and Genie are where most people consume the data you build. If Gold tables have unclear names, no comments and mixed grains, dashboards are hard to build and Genie gives wrong answers. The exam's platform domain covers Databricks SQL and AI/BI features, and interviewers like to hear that you design tables <b>for their consumers</b>.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 AI/BI dashboards and Genie</b><br>
| | AI/BI dashboard | Genie space |<br>|---|---|---|<br>| For | Fixed, curated reports and KPIs | Ad hoc questions in plain language |<br>| Built from | <b>Datasets</b> (SQL queries or tables) + <b>widgets</b> (charts, counters, tables) + <b>filters</b> | Tables + <b>instructions</b> + example SQL queries |<br>| Runs on | A SQL warehouse | A SQL warehouse |<br>| Sharing | Publish, share with users/groups, schedule email snapshots | Share the space with users/groups |<br>| Security | Viewers' or publisher's credentials (configurable). Unity Catalog permissions apply | Users' own Unity Catalog permissions |<br><br>
What makes both work: Gold tables with <b>clear names</b>, <b>column comments</b>, a single clear <b>grain</b>, consistent units, and no ambiguous columns.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A retail company built a sales dashboard with a revenue trend, top categories and a region filter on top of the <code>sales_daily</code> Gold table. Store managers then kept asking follow-up questions such as "which category grew fastest in the South last month?". A Genie space on the same table, with instructions like "revenue means the revenue column in USD" and two example queries, answered those questions directly, and the analysts stopped receiving ad hoc requests.
</div>

## Setup: make the Gold table consumer-ready

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Makes sure the <code>sales_daily</code> table from lesson 06 exists, adds comments to every column, and creates a monthly view that dashboards and Genie can use.<br><br>
<b>Why it matters</b><br>Comments are what Genie reads to understand your data, and what dashboard authors see in the catalog. A view with a clear grain (one row per month, region and category) avoids double counting.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Column comments shown by <code>DESCRIBE TABLE</code>, and a <code>sales_monthly</code> view with one row per month, region and category.
</div>

In [0]:
DB = "workspace.platform"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
from pyspark.sql import functions as F

TABLE = f"{DB}.sales_daily"
if not spark.catalog.tableExists(TABLE):
    days = spark.sql("SELECT explode(sequence(date'2024-09-01', date'2024-11-29', interval 1 day)) AS sale_date")
    (days.crossJoin(spark.createDataFrame([("North",), ("South",), ("West",)], "region STRING"))
         .crossJoin(spark.createDataFrame([("Electronics",), ("Furniture",), ("Books",), ("Toys",)], "category STRING"))
         .withColumn("orders", (F.abs(F.hash("sale_date", "region", "category")) % 50 + 10).cast("int"))
         .withColumn("revenue", F.round(F.col("orders") * (F.abs(F.hash("category")) % 80 + 20) * (1 + F.dayofweek("sale_date") / 20), 2))
         .write.mode("overwrite").saveAsTable(TABLE))

comments = {
    "sale_date": "Calendar date of the sales (one row per date, region and category)",
    "region": "Sales region: North, South or West",
    "category": "Product category",
    "orders": "Number of orders",
    "revenue": "Revenue in USD, after discounts, before tax",
}
spark.sql(f"COMMENT ON TABLE {TABLE} IS 'Daily sales per region and category. Grain: sale_date x region x category.'")
for col, text in comments.items():
    spark.sql(f"ALTER TABLE {TABLE} ALTER COLUMN {col} COMMENT '{text}'")

spark.sql(f"""
    CREATE OR REPLACE VIEW {DB}.sales_monthly
    COMMENT 'Monthly sales per region and category. Grain: month x region x category.'
    AS SELECT date_trunc('month', sale_date) AS month, region, category,
              sum(orders) AS orders, round(sum(revenue), 2) AS revenue
       FROM {TABLE}
       GROUP BY 1, 2, 3
""")
spark.sql(f"DESCRIBE TABLE {TABLE}").show(truncate=False)
spark.table(f"{DB}.sales_monthly").orderBy("month", "region", "category").show(5)

## 1. The dashboard datasets (queries)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Defines and runs the SQL queries that will back the dashboard: a trend, a category breakdown, KPIs, and a region filter source.<br><br>
<b>Why it matters</b><br>In AI/BI dashboards you first define <b>datasets</b> (SQL), then build widgets on them. Testing the queries here, against the same tables, catches mistakes before they reach a chart. Keep heavy logic in Gold tables, and keep dashboard queries simple.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Four small result sets: weekly revenue, revenue by category, a KPI row, and the list of regions.
</div>

In [0]:
datasets = {
    "weekly_revenue": f"SELECT date_trunc('week', sale_date) AS week, region, round(sum(revenue), 0) AS revenue FROM {TABLE} GROUP BY 1, 2",
    "category_revenue": f"SELECT category, region, round(sum(revenue), 0) AS revenue, sum(orders) AS orders FROM {TABLE} GROUP BY 1, 2",
    "kpis": f"""SELECT round(sum(revenue), 0) AS total_revenue, sum(orders) AS total_orders,
                       round(sum(revenue) / sum(orders), 2) AS avg_order_value FROM {TABLE}""",
    "regions": f"SELECT DISTINCT region FROM {TABLE} ORDER BY region",
}
for name, sql in datasets.items():
    print(f"--- {name}")
    spark.sql(sql).show(4)

## 2. Build the dashboard (UI steps)

Follow these steps in your workspace. The SQL comes from section 1.

1. Sidebar → **Dashboards** → **Create dashboard**
2. **Data** tab → **Create from SQL** → paste `weekly_revenue` → rename the dataset. Repeat for `category_revenue` and `kpis`
3. **Canvas** tab → add a visualization:
   - **Line chart** on `weekly_revenue`: X = `week`, Y = `sum(revenue)`, color by `region`
   - **Bar chart** on `category_revenue`: X = `category`, Y = `sum(revenue)`
   - **Counters** on `kpis`: total revenue, total orders, average order value
4. Add a **filter** widget on the field `region`, applied to both chart datasets
5. Optionally add a **parameter**: in a dataset's SQL use `WHERE sale_date >= :start_date`, then add a date parameter widget
6. Use **AI assistant** in the canvas ("show revenue by category as a bar chart") to draft widgets from plain language
7. **Publish** → choose whether viewers use your credentials (embedded) or their own
8. **Share** with users or groups, and **Schedule** an email snapshot (for example every Monday at 8:00)

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Dashboards, sharing and schedules work in Free Edition on the serverless SQL warehouse, within quotas.
</div>

## 3. Find your dashboards with the SDK

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists the AI/BI (Lakeview) dashboards in the workspace with the Databricks SDK.<br><br>
<b>Why it matters</b><br>Dashboards are workspace objects that can be listed, exported and deployed by code, for example as part of an Asset Bundle, so they move from dev to prod like pipelines do.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The dashboards you created, with their names and paths (or an empty list).
</div>

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
try:
    for d in w.lakeview.list():
        print(f"{d.display_name:<40} {d.path}  state={d.lifecycle_state}")
except Exception as e:
    print("dashboards not listable here:", type(e).__name__)

## 4. Create a Genie space (UI steps)

1. Sidebar → **Genie** → **New**
2. Add the data: `workspace.platform.sales_daily` and the view `workspace.platform.sales_monthly`
3. Choose the SQL warehouse
4. Add **instructions** (General instructions box), for example:
   - "Revenue means the `revenue` column, in USD. Never sum it across currencies."
   - "Weeks start on Monday. 'Last month' means the previous full calendar month."
   - "Use `sales_monthly` for monthly questions, and `sales_daily` for daily or weekly ones."
5. Add **example SQL queries** with a description, such as the `category_revenue` query from section 1, titled "Revenue by category and region"
6. Ask questions: "What was the revenue by region in November 2024?", "Which category grew the most from October to November?"
7. Check the **generated SQL** behind each answer. If one is wrong, improve the instructions or add an example, rather than accepting it

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ What makes Genie accurate</b><br>
Good column comments, one clear grain per table, business terms explained in the instructions, and a few trusted example queries. Genie can only be as good as the semantic information you give it.
</div>

## 5. Test the questions you expect Genie to answer

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the "trusted SQL" answers to the questions above, so you can compare Genie's answers with known-correct results.<br><br>
<b>Why it matters</b><br>Treat Genie like any other consumer: have test questions with known answers, and check them whenever the tables or instructions change.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Revenue by region for November 2024, and the month-over-month growth by category from October to November.
</div>

In [0]:
spark.sql(f"""
    SELECT region, round(sum(revenue), 0) AS revenue
    FROM {TABLE}
    WHERE sale_date >= '2024-11-01' AND sale_date < '2024-12-01'
    GROUP BY region ORDER BY revenue DESC
""").show()

spark.sql(f"""
    WITH m AS (SELECT category, month, sum(revenue) AS revenue FROM {DB}.sales_monthly GROUP BY category, month)
    SELECT category,
           round(max(CASE WHEN month = '2024-10-01' THEN revenue END), 0) AS oct,
           round(max(CASE WHEN month = '2024-11-01' THEN revenue END), 0) AS nov,
           round(100 * (max(CASE WHEN month = '2024-11-01' THEN revenue END)
                      / max(CASE WHEN month = '2024-10-01' THEN revenue END) - 1), 1) AS growth_pct
    FROM m GROUP BY category ORDER BY growth_pct DESC
""").show()

## Under the hood

```
Gold tables / views (Unity Catalog, comments, grain)
        │
        ├──▶ Dashboard datasets (SQL) ──▶ widgets, filters, parameters ──▶ published dashboard
        │        run on a SQL warehouse, results cached, scheduled snapshots by email
        │
        └──▶ Genie space: tables + instructions + example SQL
                 question ──▶ LLM generates SQL using metadata ──▶ runs on the warehouse ──▶ answer + SQL
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> every dashboard refresh and Genie answer is a SQL query on the warehouse, visible in Query History.

**Permissions:** viewers need access to the data through Unity Catalog, unless the dashboard is published with embedded credentials, in which case queries run as the publisher. Genie always uses the asking user's own permissions.

In [0]:
spark.sql(f"DESCRIBE TABLE EXTENDED {TABLE}").filter("col_name = 'Comment'").show(truncate=False)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: dashboard numbers double counted</b><br>
The dataset mixes grains (for example joining daily rows to monthly targets). Build each dataset on one clear grain.<br><br>
<b>⛔ Problem: Genie misunderstands a business term</b><br>Add it to the instructions, improve the column comment, or add an example query using it.<br><br>
<b>⛔ Problem: viewers see errors instead of charts</b><br>They lack <code>SELECT</code> on the tables and the dashboard uses their credentials. Grant access, or publish with embedded credentials.<br><br>
<b>⛔ Problem: slow dashboard</b><br>Heavy logic in dataset SQL. Pre-aggregate into Gold tables, and check the warehouse's scaling (lesson 06).<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What are AI/BI dashboards in Databricks?</b><br>
Dashboards built from SQL datasets and visual widgets, with filters and parameters, running on a SQL warehouse. They can be published with embedded or viewer credentials, shared with users and groups, and scheduled for email delivery.<br><br>

<b>🎤 2. What is a Genie space?</b><br>
A natural-language interface over selected Unity Catalog tables. Users ask questions, Genie generates and runs SQL on a warehouse, and returns the answer with the SQL. Authors improve it with instructions and example queries.<br><br>

<b>🎤 3. What can a data engineer do to make Genie and dashboards accurate?</b><br>
Provide clean Gold tables with clear names, a single grain, consistent units and column comments. Add views for common aggregations. Document business definitions in Genie instructions, add trusted example SQL, and test known questions.<br><br>

<b>🎤 4. How do permissions work for dashboards?</b><br>
Data access is governed by Unity Catalog. A published dashboard can run queries with the publisher's embedded credentials, or with each viewer's own credentials, in which case viewers need <code>SELECT</code> on the underlying data.<br><br>

<b>🎤 5. Why keep heavy transformations out of dashboard queries?</b><br>
Every refresh re-runs them on the warehouse, which is slow and costly, and logic gets duplicated across dashboards. Computing it once in Gold tables is faster, cheaper and consistent.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Business users want to ask questions about sales data in plain English and get answers based on governed tables. Which Databricks feature fits best?</b><br>
A. Delta Sharing<br>
B. A Genie space<br>
C. A job cluster<br>
D. Auto Loader<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. A published dashboard must show data to viewers who do not have <code>SELECT</code> permission on the underlying tables. How should it be published?</b><br>
A. With embedded (publisher's) credentials<br>
B. With each viewer's credentials<br>
C. It's impossible<br>
D. By copying the data into a CSV file<br>
<details><summary><b>Show answer</b></summary><b>A.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Add a dataset <code>top_categories_by_region</code> (rank categories within each region by revenue) and add it to your dashboard as a table widget</li><li>Add a date-range parameter to the weekly revenue dataset</li><li>In your Genie space, add an instruction that defines "best region" as the region with the highest revenue, and test the question "What was the best region in October 2024?"</li><li>Write the trusted SQL answer to that question and compare it with Genie's</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Dataset for the table widget
spark.sql(f"""
    SELECT region, category, round(sum(revenue), 0) AS revenue,
           rank() OVER (PARTITION BY region ORDER BY sum(revenue) DESC) AS rank_in_region
    FROM {TABLE} GROUP BY region, category ORDER BY region, rank_in_region
""").show()

# 2. Weekly revenue with parameters (paste into the dashboard dataset):
print(f"""SELECT date_trunc('week', sale_date) AS week, region, round(sum(revenue), 0) AS revenue
FROM {TABLE}
WHERE sale_date BETWEEN :start_date AND :end_date
GROUP BY 1, 2""")

# 4. Trusted answer for "best region in October 2024"
spark.sql(f"""
    SELECT region, round(sum(revenue), 0) AS revenue FROM {TABLE}
    WHERE sale_date >= '2024-10-01' AND sale_date < '2024-11-01'
    GROUP BY region ORDER BY revenue DESC LIMIT 1
""").show()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>AI/BI dashboards: SQL datasets → widgets, filters, parameters → publish, share, schedule</li><li>Genie: natural-language questions over governed tables, improved with instructions and example SQL</li><li>Both run on SQL warehouses and respect Unity Catalog permissions (or embedded credentials for dashboards)</li><li>Data engineers make them work: clear Gold tables, one grain, comments, views, documented business terms</li><li>Test Genie and dashboards against trusted SQL answers</li></ul>
</div>

| Task | Where |
|---|---|
| Create a dashboard | Dashboards → Create dashboard |
| Add data | Data tab → Create from SQL / table |
| Share and schedule | Publish → Share / Schedule |
| Create a Genie space | Genie → New → tables, instructions, examples |
| Document data | `COMMENT ON TABLE`, `ALTER COLUMN ... COMMENT` |

## Git commit

```
git add 04_databricks/07_dashboards_and_genie.ipynb
git commit -m "add 04_07 dashboards and genie notebook"
```